In [1]:
#loading filtered rna data
import anndata
from anndata import AnnData

filtered_Rdata = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/RNA/filtered_Rdata.h5ad")
filtered_Rdata.obs

,developmental_stage,dataset,zebrafish_anatomy_ontology_class,zebrafish_anatomy_ontology_class_coarse,timepoint,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_nc,pct_counts_nc
AAACAGCCACCTAAGC-1_1,15somites,TDR118,epidermis,epidermis,16hpf,2317,6522.0,160.0,2.453235,726.0,11.131555
AAACAGCCAGGGAGGA-1_1,15somites,TDR118,pronephros,pronephros,16hpf,2319,6100.0,245.0,4.016393,1051.0,17.229508
AAACAGCCATAGACCC-1_1,15somites,TDR118,hindbrain,hindbrain,16hpf,3467,12581.0,779.0,6.191877,2542.0,20.205071
AAACATGCAAACTCAT-1_1,15somites,TDR118,spinal_cord,spinal_cord,16hpf,2145,5642.0,265.0,4.696916,1075.0,19.053527
AAACATGCAAGGACCA-1_1,15somites,TDR118,neural_optic2,neural_optic,16hpf,838,2691.0,181.0,6.726124,732.0,27.201784
...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGTTCCCTCAGT-1_7,10somites,TDR128,tail_bud,tail_bud,14hpf,1281,3079.0,114.0,3.702501,572.0,18.577460
TTTGTTGGTACCTTAC-1_7,10somites,TDR128,lateral_plate_mesoderm,lateral_plate_mesoderm,14hpf,2441,6276.0,179.0,2.852135,962.0,15.328235
TTTGTTGGTATTGAGT-1_7,10somites,TDR128,neural_posterior,neural_posterior,14hpf,1164,2377.0,74.0,3.113168,217.0,9.129154
TTTGTTGGTGCGCGTA-1_7,10somites,TDR128,endocrine_pancreas,endocrine_pancreas,14hpf,811,1750.0,54.0,3.085714,182.0,10.400000


In [6]:
print("Max percentage of MT genes in RNA data:", filtered_Rdata.obs["pct_counts_mt"].max())
print("Min percentage of MT genes in RNA data:", filtered_Rdata.obs["pct_counts_mt"].min())
#cheking how many cells have more than 10% of MT genes
num_cells_high_mt = (filtered_Rdata.obs["pct_counts_mt"] > 10).sum()
print("Number of cells with more than 10% MT genes:", num_cells_high_mt)    

Max percentage of MT genes in RNA data: 16.584500098599882
Min percentage of MT genes in RNA data: 0.11827321111768185
Number of cells with more than 10% MT genes: 223


## Pseudobulking the RNA-Seq data
- creating new matrix of raw counts of RNA-Seq data 
- creating df for metadata 
- aggregating the counts by 1) cell type, 2) cell type and developmental stages
    - aggregation equation: total count of a gene / total count of the matrix   
This normalizes each gene's count by the total counts within each cell type group, giving a proportional representation rather than raw counts.

In [2]:
#creating a aggregation function to sum the counts for each group then divide by the total counts
def agg_fun(mat, meta, fun="sum", gene_names=None):
    """Aggregate count matrix by metadata groups (e.g., cell type + developmental time)."""
    import pandas as pd
    import numpy as np
    
    meta = pd.Series(meta)
    
    if fun == "sum":
        def agg_f(x):
            col_sums = mat[x, :].sum(axis=0)  # Sum counts for each column (gene) in selected rows
            total = mat[x, :].sum()           # Total sum of counts in selected rows
            return col_sums / total
        
        # Apply aggregation function to each group
        groups = meta.unique()
        agg_result = pd.DataFrame(
            [agg_f(meta == group) for group in groups],
            index=groups,
            columns=gene_names if gene_names is not None else None,
        )
        return agg_result

In [3]:
# Convert sparse matrix to dense if needed & extract metadata
rna_ct_matrix = filtered_Rdata.X.toarray()
cell_types = filtered_Rdata.obs["zebrafish_anatomy_ontology_class"].astype(str)
dev_times = filtered_Rdata.obs["developmental_stage"].astype(str)
gene_names = filtered_Rdata.var_names


In [4]:
#Aggregate by cell type
agg_rna_ct = agg_fun(rna_ct_matrix, cell_types, fun="sum", gene_names=gene_names)
agg_rna_ct.head()


,ptpn12,phtf2,phtf2.1,CU856344.1,si:zfos-932h1.3,mansc1,lrp6,dusp16,crebl2,gpr19,...,mt-nd4,NC-002333.16,NC-002333.15,NC-002333.8,mt-nd5,mt-nd6,NC-002333.21,mt-cyb,NC-002333.22,NC-002333.11
epidermis,0.000004,1.700115e-06,0.000002,9.925015e-08,0.000001,0.000000e+00,0.000005,0.000013,0.000010,0.000013,...,0.001039,0.000026,0.000014,2.899564e-07,0.000524,0.000140,3.796276e-07,0.001557,9.463677e-07,4.628431e-07
pronephros,0.000001,9.911735e-07,0.000002,1.457126e-07,0.000003,1.954986e-07,0.000009,0.000015,0.000009,0.000018,...,0.001060,0.000027,0.000014,0.000000e+00,0.000546,0.000132,1.531938e-06,0.001643,1.390369e-06,1.163442e-06
hindbrain,0.000002,1.576311e-06,0.000005,1.954849e-07,0.000001,0.000000e+00,0.000008,0.000013,0.000013,0.000007,...,0.001092,0.000021,0.000013,4.018211e-07,0.000487,0.000143,8.266328e-07,0.001618,1.444670e-06,8.208058e-07
spinal_cord,0.000003,1.796678e-06,0.000004,2.554442e-07,0.000002,0.000000e+00,0.000006,0.000011,0.000009,0.000011,...,0.001061,0.000018,0.000014,3.849728e-07,0.000512,0.000141,3.162579e-07,0.001595,1.113618e-06,5.847453e-07
neural_optic2,0.000002,1.957862e-06,0.000005,9.266277e-08,0.000003,0.000000e+00,0.000006,0.000012,0.000009,0.000016,...,0.001036,0.000024,0.000014,2.994966e-07,0.000521,0.000143,1.099168e-06,0.001598,1.081272e-06,6.202779e-07


In [ ]:
#Aggregate by cell type and developmental stages
meta_ct_time = cell_types + "__" + dev_times #Build modified meta vector: cell type + developmental time

agg_rna_ct_time = agg_fun(rna_ct_matrix, meta_ct_time, fun="sum", gene_names=gene_names)
agg_rna_ct_time.head()

,ptpn12,phtf2,phtf2.1,CU856344.1,si:zfos-932h1.3,mansc1,lrp6,dusp16,crebl2,gpr19,...,mt-nd4,NC-002333.16,NC-002333.15,NC-002333.8,mt-nd5,mt-nd6,NC-002333.21,mt-cyb,NC-002333.22,NC-002333.11
epidermis__15somites,0.000003,1.305613e-06,0.000003,3.334782e-07,0.000001,0.0,0.000009,0.000014,0.000008,3.052749e-06,...,0.000921,0.000016,0.000022,3.937444e-07,0.000484,0.000214,8.348790e-07,0.001317,8.404127e-07,0.000000e+00
pronephros__15somites,0.000004,7.668370e-07,0.000003,0.000000e+00,0.000004,0.0,0.000020,0.000009,0.000006,1.196225e-06,...,0.000954,0.000013,0.000033,0.000000e+00,0.000531,0.000225,5.761124e-07,0.001371,2.953837e-06,9.440679e-07
hindbrain__15somites,0.000003,2.099866e-06,0.000007,6.796223e-07,0.000002,0.0,0.000011,0.000013,0.000005,1.666828e-06,...,0.000982,0.000013,0.000023,0.000000e+00,0.000543,0.000235,2.059240e-06,0.001421,1.471980e-06,2.484611e-06
spinal_cord__15somites,0.000005,2.706247e-06,0.000006,2.824654e-07,0.000003,0.0,0.000007,0.000012,0.000010,9.262147e-07,...,0.000962,0.000009,0.000026,0.000000e+00,0.000530,0.000228,1.020105e-06,0.001388,9.326634e-07,6.972355e-07
neural_optic2__15somites,0.000002,2.756720e-06,0.000007,0.000000e+00,0.000005,0.0,0.000010,0.000014,0.000006,2.086196e-06,...,0.000970,0.000016,0.000030,0.000000e+00,0.000540,0.000255,2.389853e-06,0.001426,1.402613e-06,8.828379e-07


In [6]:
# Checking the aggregation results by the shape of the resulting dataframe and its columns
print(agg_rna_ct.shape)
print(agg_rna_ct.index)
agg_rna_ct.columns

print(agg_rna_ct_time.shape)
print(agg_rna_ct_time.index)
agg_rna_ct_time.columns

(39, 32057)
Index(['epidermis', 'pronephros', 'hindbrain', 'spinal_cord', 'neural_optic2',
       'neural_floor_plate', 'neural_crest2', 'PSM', 'optic_cup',
       'lateral_plate_mesoderm', 'midbrain_hindbrain_boundary2',
       'neural_telencephalon', 'differentiating_neurons', 'muscle',
       'fast_muscle', 'heart_myocardium', 'somites', 'NMPs', 'epidermis2',
       'pharyngeal_arches', 'floor_plate2', 'hemangioblasts',
       'neural_posterior', 'floor_plate', 'tail_bud', 'endoderm',
       'midbrain_hindbrain_boundary', 'neural_crest', 'neural_optic',
       'hematopoietic_vasculature', 'endocrine_pancreas', 'hatching_gland',
       'neurons', 'notochord', 'pronephros2', 'enteric_neurons', 'epidermis3',
       'epidermis4', 'neural'],
      dtype='object')
(196, 32057)
Index(['epidermis__15somites', 'pronephros__15somites', 'hindbrain__15somites',
       'spinal_cord__15somites', 'neural_optic2__15somites',
       'neural_floor_plate__15somites', 'neural_crest2__15somites',
      

Index(['ptpn12', 'phtf2', 'phtf2.1', 'CU856344.1', 'si:zfos-932h1.3', 'mansc1',
       'lrp6', 'dusp16', 'crebl2', 'gpr19',
       ...
       'mt-nd4', 'NC-002333.16', 'NC-002333.15', 'NC-002333.8', 'mt-nd5',
       'mt-nd6', 'NC-002333.21', 'mt-cyb', 'NC-002333.22', 'NC-002333.11'],
      dtype='object', length=32057)

In [33]:
#Comparing the number of combinations of cell types and developmental stages ("meta_ct_time") with the index (row) of the aggregated data frame to find out which groups are (not) present in the aggregated data frame
import numpy as np

meta_groups = set(meta_ct_time.unique())
agg_groups = set(agg_rna_ct_time.index)

print("Missing in ct + time dataframe:", meta_groups - agg_groups)
print("Number of combinationsof cell types and developmental stages:", len(meta_groups))
print("Number of groups in the ct + time dataframe:", len(agg_groups))


Missing in ct + time dataframe: set()
Number of combinationsof cell types and developmental stages: 196
Number of groups in the ct + time dataframe: 196


In [10]:
# Save the aggregated data frames to a new AnnData object
import pandas as pd
agg_rna_ct = AnnData(X=agg_rna_ct.values, obs=pd.DataFrame(index=agg_rna_ct.index), var=pd.DataFrame(index=agg_rna_ct.columns))
agg_rna_ct.write_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/RNA/celltypes/agg_rna_ct.h5ad")

agg_rna_ct_time = AnnData(X=agg_rna_ct_time.values, obs=pd.DataFrame(index=agg_rna_ct_time.index), var=pd.DataFrame(index=agg_rna_ct_time.columns))
agg_rna_ct_time.write_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Pseudobulks/RNA/celltypes_times/agg_rna_ct_time.h5ad")

## Log-transformation?
There's no need to log-transform due to the fact, that the values are proportions (0 to 1) 